# Privacy Lens · 다음 학습 (증명사진 모델, 번호판 v004)

`vision/next_steps.py` 의 단계를 칸 하나씩 부른다. **칸마다 혼자 돈다**(앞 칸 변수에 기대지 않음). 런타임이 끊기면 0번 칸 다음 그 칸만 다시 돌린다.

| 칸 | 하는 일 | GPU | 시간(T4) |
|---|---|---|---|
| 0 | 드라이브 연결·코드 받기·설치 | - | 3분 |
| 1 | 데이터 준비: Open Images 번호판, 한국 번호판 zip 풀기, 합성·90도 회전 사본, MIDV-2020 증명사진 | 필요 없음 | 20~30분 |
| 2 | **증명사진 모델 학습**(YOLO26n, 40 epoch) → `objects/face_photo_v001/` | 필요 | 40~60분 |
| 3 | 증명사진 평가: YuNet(서비스) vs 학습 모델, 처음 보는 신분증 종류 2개에서 크기·방향별 | 필요 없음 | 10~20분 |
| 4 | **번호판 v004 학습**(v003 + 90도 회전 사본, 20 epoch) → `objects/plate_v004/` | 필요 | 60분 |
| 5 | 번호판 평가: v003 vs v004, 한국(바로·누움)·해외, 돌려 찾기 끔/켬 | 있으면 빠름 | 10분 |

- 데이터: MIDV-2020(CC BY-SA 2.5, 얼굴은 생성 얼굴), Open Images(CC BY 4.0), Roboflow 한국 번호판 3종(CC BY 4.0, 드라이브 `objects/kr/`). 사진은 드라이브·런타임에만 둔다.
- 결과: 드라이브 `objects/eval_face_photo.json`, `objects/eval_plate_v004.json`. 숫자는 `vision/README.md` 에 옮겨 적는다.
- GPU 한도가 없으면 1·3번 칸은 CPU 로도 돈다(런타임 → 런타임 유형 변경 → CPU).
- 학습 기록은 드라이브 `objects/runs/<이름>/` 에 저장한다. 런타임이 끊겨도 0·1번 칸 다음 같은 학습 칸을 다시 돌리면 **끊긴 epoch 부터 이어서** 학습한다(끝난 학습은 `DONE` 표시가 있어 건너뜀).


In [ ]:
BRANCH = "feat/vision-objects"
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!GIT_TERMINAL_PROMPT=0 git pull -q --ff-only && git log --oneline -1
!pip -q install "ultralytics>=8.4" onnxruntime-gpu fiftyone
!python -m pipeline.objects download
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
info = ns.plate_data()      # Open Images 번호판·한국 zip·합성·회전 사본
ns.midv_dataset()           # MIDV-2020 → 증명사진 YOLO 데이터(배경은 Open Images 차량 사진)

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
face_model = ns.train_face_photo(epochs=40)

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
face_res = ns.eval_face_photo()   # 가장 최근 objects/face_photo_vNNN 을 쓴다

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
plate_model = ns.train_plate_rot(epochs=20)

In [ ]:
import importlib, vision.next_steps as ns; importlib.reload(ns)
plate_res = ns.eval_plate()   # objects/plate_v003 이후 모델을 모두